# Heater characterization

Follows both machines powered in the chamber, from the 30-31 Oct 2026 run: chamber temperature against low-side heater voltage, sense voltage and thermocouples. The sensor curves from `sensor_characterization` turn sense voltage into thermistor temperature.

**Probes**, from the bench setup, with channels inferred below:
- ch5: inside the chamber, used as the temperature reference.
- ch2 and ch3: on each machine's thermistor.
- ch1 and ch4: on each machine's exposed pad.
- ch8: outside the chamber.

**Assumed:** sensor resistance comes from sense voltage using a 10 kohm pull-up and exactly +5 V. The QR101B substitution test replaces this. The `Sensor (n)` numbering is assumed to match the passive sweep.

**Low-side voltage:** the sweep code used `MEAS:VOLT:DC?`, so these logs are DC readings unless the meter was set up differently.

`results_20261030-010329.txt` is an earlier powered run. It has no matching chamber reference, so nothing here uses it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cryojane_bench.logs import read_channel, read_picolog, read_results_log
from cryojane_bench.plotting import BLUE, INK, MUTED, ORANGE, apply_style
from cryojane_bench.thermistor import SteinhartHart, divider_voltage_to_resistance

apply_style()
DATA = Path("data")

sweep = read_results_log(DATA / "results_20261030-223516.txt")
pico = read_picolog(DATA / "20261031_114234.picolog").loc["2026-10-30":]
pico.columns = [column.split()[0] for column in pico.columns]  # "PMC 053x" labels are left over from an older capture.

CHAMBER = "ch5"
MACHINES = {
    "0": {"thermistor": "ch2", "exposed": "ch1"},
    "1": {"thermistor": "ch3", "exposed": "ch4"},
}
fit_summary = pd.read_csv("../sensor_characterization/results/fit_summary.csv", index_col="channel")
curves = {m: SteinhartHart(*fit_summary.loc[f"Sensor ({m})", ["sh_a", "sh_b", "sh_c"]].astype(float)) for m in MACHINES}
sense_v = {m: read_channel(sweep, f"Sensor ({m})") for m in MACHINES}
low_side_v = {m: read_channel(sweep, f"Heater ({m})") for m in MACHINES}

def thermistor_c(machine, volts):
    return curves[machine].temperature_c(divider_voltage_to_resistance(volts))

## Probes and heater state

The low-side node reads 0.3-0.7 V while the transistor conducts and about 24 V when it is off, so readings above 1 V mark off-pulses. Each machine's first off-pulse lines up with its thermistor probe going flat. Each exposed probe's rise above the chamber stays at about a quarter of its partner's as the heaters back off.

In [ ]:
first_off_pulse = {m: volts[volts > 1.0].index.min() for m, volts in low_side_v.items()}
for machine, moment in first_off_pulse.items():
    print(f"Machine {machine}: first off-pulse {moment}")

window = pico.loc["2026-10-30 23:00":"2026-10-31 11:30"].resample("2min").mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (machine, probes) in zip(axes, MACHINES.items()):
    ax.plot(window.index, window[CHAMBER], color=MUTED, label=f"Chamber ({CHAMBER})")
    ax.plot(window.index, window[probes["exposed"]], color=ORANGE, label=f"Exposed pad ({probes['exposed']})")
    ax.plot(window.index, window[probes["thermistor"]], color=BLUE, label=f"On thermistor ({probes['thermistor']})")
    ax.axvline(first_off_pulse[machine], color=INK, linestyle=":", linewidth=1.5, label="First heater off-pulse")
    ax.set_title(f"Machine {machine}")
    ax.legend(loc="lower right")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Temperature (°C)")
fig.suptitle("The thermistor probe goes flat when the heater first switches off", x=0.01, ha="left", color=INK)
fig.tight_layout()
plt.show()

## Thermistor temperature against the probes, at full power

Each window is the end of a chamber dwell. Machine 0's last window is already regulating, so it is marked. Medians reject the single-reading spikes seen while a heater switches.

The thermistor's own reading sits a steady amount below the probe taped to it: about 5 °C on machine 0 and about 2 °C on machine 1. A constant offset at constant power looks more like a thermal gradient or probe placement than a scale error, but a sense-voltage shift from heater current is not ruled out. The substitution test separates the two.

In [ ]:
WINDOWS = [
    ("2026-10-30 23:10", "2026-10-31 00:10"),
    ("2026-10-31 00:40", "2026-10-31 01:10"),
    ("2026-10-31 01:40", "2026-10-31 02:10"),
    ("2026-10-31 02:40", "2026-10-31 03:10"),
    ("2026-10-31 03:40", "2026-10-31 04:10"),
    ("2026-10-31 04:40", "2026-10-31 05:10"),
]

def dwell_table(machine):
    probes = MACHINES[machine]
    rows = []
    for start, stop in WINDOWS:
        volts = sense_v[machine].loc[start:stop].median()
        rows.append({
            "chamber_c": pico[CHAMBER].loc[start:stop].mean(),
            "exposed_probe_c": pico[probes["exposed"]].loc[start:stop].mean(),
            "thermistor_probe_c": pico[probes["thermistor"]].loc[start:stop].mean(),
            "sense_v": volts,
            "thermistor_c": float(thermistor_c(machine, volts)),
            "full_power": pd.Timestamp(stop) < first_off_pulse[machine],
        })
    table = pd.DataFrame(rows)
    table["thermistor_minus_probe_c"] = table.thermistor_c - table.thermistor_probe_c
    return table

dwells = {machine: dwell_table(machine) for machine in MACHINES}
for machine, table in dwells.items():
    print(f"Machine {machine}")
    print(table.round(2).to_string(index=False), end="\n\n")

fig, ax = plt.subplots(figsize=(9, 4))
for machine, color in (("0", BLUE), ("1", ORANGE)):
    table = dwells[machine]
    ax.plot(table.chamber_c, table.thermistor_probe_c, "o--", color=color, markersize=7, markeredgecolor="#fcfcfb", linewidth=1.5, label=f"Machine {machine}: probe on thermistor")
    ax.plot(table.chamber_c, table.thermistor_c, "o-", color=color, markersize=7, markeredgecolor="#fcfcfb", label=f"Machine {machine}: thermistor, from its own curve")
ax.set(xlabel="Chamber temperature (°C)", ylabel="Temperature (°C)")
ax.set_title("The thermistor reads colder than the probe on it, by a steady amount")
ax.legend()
fig.tight_layout()
plt.show()

## Regulation point

Once a machine regulates, its sense voltage holds still. Machine 0 has 57 minutes of data in this state. The GDM log ended at 05:21, so machine 1 has one settled block, about a minute after its first off-pulse, and is less certain.

The probe range covers the regulated span, while the probe stays at least 3 °C above the chamber so the heater still has work to do.

In [ ]:
REGULATED_BLOCKS = {
    "0": ("2026-10-31 04:21:00", "2026-10-31 05:17:00"),
    "1": ("2026-10-31 05:19:45", "2026-10-31 05:20:05"),
}
MIN_RISE_C = 3.0

rows = {}
for machine, (start, stop) in REGULATED_BLOCKS.items():
    probes = MACHINES[machine]
    volts = sense_v[machine].loc[start:stop].median()
    at_onset = pico.loc[first_off_pulse[machine] : first_off_pulse[machine] + pd.Timedelta(minutes=1)].mean()
    probe = pico[probes["thermistor"]].loc[start:]
    held = probe[probe - pico[CHAMBER].loc[start:] >= MIN_RISE_C]
    rows[f"Machine {machine}"] = {
        "sense voltage (V)": volts,
        "sensor resistance (kohm)": float(divider_voltage_to_resistance(volts)) / 1e3,
        "thermistor, from its own curve (C)": float(thermistor_c(machine, volts)),
        "probe on thermistor while regulating, median (C)": held.median(),
        "probe on thermistor while regulating, min (C)": held.min(),
        "probe on thermistor while regulating, max (C)": held.max(),
        "exposed pad at first off-pulse (C)": at_onset[probes["exposed"]],
        "chamber at first off-pulse (C)": at_onset[CHAMBER],
    }
print(pd.DataFrame(rows).round(2).to_string())

## Pad temperature at the recommended cryostat range

The CryoJane manual calls for a -25 to -30 °C cryostat. Both machines start regulating only above that range, so in normal use their heaters run at full power and the pad temperature follows the cryostat.

In [ ]:
CRYOSTAT_C = [-30.0, -25.0]
rows = []
for machine, table in dwells.items():
    full_power = table[table.full_power]
    for cryostat in CRYOSTAT_C:
        rows.append({
            "machine": machine, "cryostat_c": cryostat,
            "exposed_pad_c": np.interp(cryostat, full_power.chamber_c, full_power.exposed_probe_c),
            "thermistor_from_curve_c": np.interp(cryostat, full_power.chamber_c, full_power.thermistor_c),
            "probe_on_thermistor_c": np.interp(cryostat, full_power.chamber_c, full_power.thermistor_probe_c),
        })
print(pd.DataFrame(rows).round(2).to_string(index=False))